# From a written requirement to an evidence-linked case report
## A small specification-led development example
**13 November 2026 · Proposed specification v0.1 · Draft PR #7**

We want a reader to answer four questions without tracing a long log: **What did the requester learn? What did the authority record? What happened to the action? Which evidence is available?**

This notebook specifies a new report component before its implementation. It uses saved records from two earlier local cases. The records already exist; the component and its acceptance checks do not.

| Stage | Position today |
|---|---|
| Inspect the saved sources | Source reading completed for this draft |
| Write requirements and acceptance examples | **WE ARE HERE: proposed for review** |
| Implement the report component | Not started in this work |
| Run approved acceptance checks | Not run |
| Review results and decide about merging | Not reached |

No executable cells are included. Reading this notebook does not run a study, call an AI model or authorize a merge. This is a development example, not an experiment showing that spec-driven development improves outcomes.

**Reading route:** 1. The problem → 2. Four questions → 3. Our requirements → 4. Evidence boundary → 5. Acceptance examples → 6. Development handoff → 7. Review and next decision.

The earlier learning lesson explains the case story. This notebook explains how we could build a report from it. Technical source identities and detailed field rules follow the main explanation.

## 1. The problem: the same message, different records
**Revocation means withdrawing permission previously granted.** A requester can ask an assistant to stop an action before it happens. The request, the effective withdrawal and the confirmation received are separate events.

Our earlier study used local synthetic actions. The tool was named `send_email`, but it recorded an inert summary locally and sent no email. **S4 and S5 are schedule labels assigned to two controlled cases**, not software versions or phases of this new work.

| Saved case | Requester saw | Later reviewer can inspect |
|---|---|---|
| S4: withdrawal recorded; completion message withheld | Unconfirmed at observation end | A withdrawal event, a later attempted action, a denial and an empty local-effect record |
| S5: delivery of withdrawal deliberately withheld | Unconfirmed at observation end | Active authority state, an attempted action and a recorded local effect |

The new report should keep these perspectives separate. A later reviewer has records the requester did not receive. Neither screen access nor a trace tells us what a person understood.

**Learning prompt:** What extra record helps distinguish the two cases? Look for the authority and action evidence, rather than asking the screen message to answer every question.

## 2. Four questions, four places in the report
```text
Saved review packet
        │
        ├── Requester: What message reached the caller?
        ├── Authority: What state and transition were recorded?
        ├── Action:   What attempt, result and local effect were recorded?
        └── Evidence: What is present, missing, conflicting or unusable?
                         ↓
             Reader's bounded conclusion
```

| Panel | Plain-language question | Technical distinction |
|---|---|---|
| Requester | What did the requester learn? | Caller observation is separate from authority state. |
| Authority | What permission state was recorded? | A snapshot and a transition describe different facts. |
| Action | What happened to the selected action? | A denied attempt and cumulative local effects are separate facts. |
| Evidence | How much of the declared packet do we have? | Packet coverage does not establish complete system coverage. |

Each panel needs its own sources and uncertainty. Avoid a single green “success” badge that hides a missing panel.

**Where LangChain and LangGraph fit:** the historical framework path uses real agent/tool orchestration, approval interrupts and checkpoint resume. Its model replies are deterministic scripts; approval is supplied by the test fixture. The new report reads saved outputs. It does not rerun that workflow or demonstrate live-model judgment.

## 3. Our written promise
### REQ-REPORT-01: separate the four conclusions
The report presents requester status, authority state, action outcome and evidence coverage separately. Every factual conclusion names its source record and scope. Missing or conflicting records remain visible.

| Requirement | Acceptance obligation |
|---|---|
| REQ-02 · Scope | State the selected local case, protected action and saved observation boundary. |
| REQ-03 · Identity | Bind each assertion to packet, source case, root/epoch and request/action identities where available. Do not join records merely because a label looks familiar. |
| REQ-04 · Evidence | Separate recorded facts, classifications derived from them and unavailable information. |
| REQ-05 · Uncertainty | The requester's unconfirmed state cannot substitute for authority or action evidence. |
| REQ-06 · Coverage | Compare available evidence with the declared packet inventory. Describe system-wide coverage as unestablished. |
| REQ-07 · Conflict | Preserve incompatible assertions, identify the affected conclusion and request resolving evidence. |
| REQ-08 · Provenance | Distinguish original historical records from constructed teaching variants. Preserve omissions and transformations. |

These are proposed component requirements. A written requirement is not its implementation or its proof. The approach is inspired by spec-driven development for agentic software engineering; the component remains ordinary, inspectable deterministic software.

## 4. Define what the component may read
The original records contain repeated copies of evidence. The same event can appear in the final sink, barrier snapshots and worker traces. Removing one copy would leave the answer visible elsewhere.

The proposed extractor therefore makes a **derived review packet** from an explicit allowlist. Historical files remain untouched. Each teaching variant receives a separate identity and a transformation record. The report component receives only that packet, not the whole archive, hidden answer key or schedule explanation.

```text
Frozen original archive (unchanged)
                 ↓ declared extraction and omission record
Derived packet supplied to the report
                 ↓ deterministic classification and rendering
Four-panel report with source locators

Reviewer reference record stays separate from component input.
```

| Allowed input category | Original field or location | Required interpretation |
|---|---|---|
| Requester observation | `/caller_state` | Message state at the saved observation end |
| Authority snapshot | `/sink/authority` | State at the saved snapshot, not automatically at every prior instant |
| Authority/action events | Selected entries of `/sink/events` | Preserve each entry's sequence, kind and data; select by explicit event identity |
| Local effects | `/sink/effects` | Recorded effects within the supplied inventory; an empty supplied list alone does not establish global absence |
| Packet metadata | Newly authored manifest | Source identity, allowed fields, omissions, boundaries, transformations and unresolved coverage |

`/workers`, `/barriers`, `/execute_response`, `/relay_trace`, schedule descriptions and other fields are outside this component's input. They remain available to reviewers in the original archive. The selection does not erase the original information.

**No hidden fill-ins:** incomplete packets stay incomplete. The report cannot look up S4 or S5 and supply the expected answer from memory.

### 4A. Precise identity and allowed-field contract
A proposed packet has `packet_id`, `schema_version`, `source_case`, `source_path`, `source_archive_sha256`, `source_member_sha256`, `input_revision`, `packet_boundary`, `fixture_kind`, `transformations`, `expected_categories`, `packet_sha256`, `source_verification_receipt` and the four allowed evidence categories. A missing category is represented explicitly as unavailable, not converted to an empty list.

Each evidence item retains its original JSON pointer, source member identity and original value. Every join key is prefixed by packet ID, source case and archive/member digest because request IDs, roots and sequence numbers recur across cases. Each derived statement retains the input item IDs and the rule used to interpret them. Original event sequence numbers are meaningful within this local authority's event stream; they are not cross-machine timestamps.

| Identity | Matching rule for this POC |
|---|---|
| Source | Exact archive/member digests and pinned repository revision |
| Case | Separate source case and packet ID; never combine S4 and S5 into one history |
| Authority | `rootR`, with each snapshot/event's own epoch meaning retained |
| Withdrawal | `withdrawal-1`; transition records name `withdrawn_epoch=7`, `new_epoch=8` |
| Action attempt | `resume-execute`, grant `child-1`, scope `summary:task1`, ticket `ticket-1`, when present in the supplied receipt |
| Effect | `effect_key=summary:task1`, root and epoch, linked local `event_seq` |

Fields not present in a source item must not be invented. For example, the denial result itself has no root; its request ID and same-stream receipt provide the proposed linkage. The implementation review must make that join explicit. Retain the same-case receipt and grant/bootstrap context needed for this derived association; never present joined identity as directly stored in the denial.

The proposed local extractor checks original bytes against the pinned archive/member digests before deriving packets. It emits a separate source-verification receipt with checked identities and result. The report consumes that receipt and its packet binding; it cannot recompute an original archive digest from selected fields. Missing or failed verification remains explicit. This assumes the declared local extractor/custody boundary; the receipt is not a cryptographic attestation of truth. A8a exercises this extraction boundary, while A8b and A8c exercise report-input identity handling.

An epoch change is meaningful evidence, not an identity mismatch by itself. A conflicting or unrelated root, unbound attempt, invalid digest or missing required source identity makes the affected conclusion unbound. Preserve unaffected observations separately.

For A1, the original event allowlist is S4 sequence 1, 3, 5, 6, 7, 8 and 9. For A2, it is S5 sequence 1, 3, 5, 6 and 7. These retain bootstrap/grant/admission context and the relevant withdrawal/action events. The requester state, final authority snapshot and effect inventory are admitted separately. Missing variants remove disallowed event kinds and categories entirely; no raw remainder is passed through.

`expected_categories` is a declared input contract, not a claim that the archive contains everything the real world could reveal. Report **available for this packet** only after checking all declared categories and required identities. Report semantic conflict separately from availability.

## 5. Acceptance examples fixed before implementation
**These are planned examples and expected behavior. None has been executed as a new component check.** A1 and A2 use selected original records. A3–A8 are proposed constructed variants or malformed-input examples, not additional observed study cases.

| ID | Supplied evidence | Expected report |
|---|---|---|
| A1 · Saved S4 | Original selected evidence | Caller unconfirmed; final authority inactive at epoch 8; withdrawal at sequence 7, later action receipt 8 and denied/revoked 9; zero entries in the supplied local-effect inventory. State this bounded ordering and source scope. |
| A2 · Saved S5 | Original selected evidence | Caller unconfirmed; final authority active at epoch 7; action receipt 6 and effect event 7; one matching local synthetic effect. The active state and positive effect record support the account. |
| A3 · Caller only | Only requester state and provenance retained | Caller unconfirmed. Authority and action unknown; missing categories listed. Request authority transition/snapshot and matching action/effect records. |
| A4 · No action record | S4 authority evidence retained; action and effect evidence omitted | Report recorded withdrawal and authority snapshot. Action outcome remains unknown. A withdrawal record does not establish that no effect occurred. |
| A5 · Denial without effect inventory | S4 linked attempt/denial retained; effect inventory omitted | Report the denied attempt. Cumulative local-effect coverage remains unknown. Do not erase the possibility of an earlier effect. |
| A6 · Positive effect, incomplete inventory | One matching S5 effect entry retained; inventory completeness declaration withheld | Report at least one recorded effect. Total count and overall coverage remain unknown. Missing completeness does not erase positive evidence. |
| A7 · Constructed contradiction | Two incompatible assertions for the same identity and declared boundary, within a labeled constructed packet | Display both and mark the affected conclusion conflicted. Request the original authority/event record and provenance clarification. Do not silently prefer a source. |
| A8 · Unbound input | Separately specified malformed source identity, digest mismatch or unrelated root/attempt | Mark affected evidence unusable or unbound; preserve the reason and unaffected observations. No confident joined conclusion or silent repair. |

**A7 construction constraint:** a later inactive snapshot and an earlier effect are compatible. Construct A7 from the S5 authority snapshot by adding a second, explicitly synthetic assertion for the same case/root/epoch and terminal boundary with `active=0`, alongside the original `active=1`. Give each assertion its own item ID and provenance. This is fabricated input for a future check, not a historical finding. Do not alter the original member or claim the derived packet has its original digest.

**A8 scope constraint:** A8a declares an original member digest that does not match the supplied original bytes; A8b changes the derived effect root to an unrelated root while retaining the original action receipt; A8c omits the mandatory source-member digest. Record every modification and give each variant its own digest. These are three separately named planned fixtures. An epoch transition from 7 to 8 is expected in S4 and must not be treated as a mismatch.

All examples must preserve warranted distinctions and source links. A count of passing examples cannot compensate for one unsupported confident conclusion. Passing this set would establish behavior on this set, not general evidence sufficiency.

### 5A. What readers should see
**Illustrative layout, not generated component output:**

| Requester | Authority | Action | Evidence |
|---|---|---|---|
| Unconfirmed at observation end | Withdrawal recorded; final snapshot inactive | Later attempt denied; supplied local-effect inventory empty | Named original records available; local scope only |

Below the summary, provide four short explanations, a timeline and an equivalent table. Place exact source pointers beside each statement. A reader can inspect the technical evidence without leaving the main lesson.

For an incomplete packet, replace unsupported findings with **Unknown from the supplied packet**. Then name the next useful evidence request. For example: “Obtain the effect inventory covering the declared local observation boundary.” Do not promise that this request will exist or settle the case.

**Pause and predict:** A denial record arrives, but the effect inventory is unavailable. What can the report say? It can describe that denied attempt. It cannot establish the cumulative history of effects from the denial alone.

## 6. Development handoff: an inspectable AI-team process
The next cycle should leave a record another reader can follow:

```text
Accepted specification revision
             ↓
Exact implementer assignment → returned code and change account
             ↓
Separate source review → concerns → corrections or retained limits
             ↓
Mark's approval for specific checks
             ↓
Exact execution record → requirement-by-requirement result
             ↓
Separate merge decision
```

No stage is completed simply by drawing this diagram. An AI review is not independent human validation. This notebook presents a development method to practice, not a new framework or a validated research instrument.

### Proposed implementer assignment · NOT DISPATCHED
The text below is prepared for a future assignment. It has not been sent to an implementer, and no returned implementation is implied.

> **Assignment POC-REPORT-01: implement the frozen-record case-report component.**
>
> Use the approved specification and acceptance examples at their pinned revision. Read only the declared derived-packet fields. Produce a deterministic report that separates requester status, authority state, action outcome and evidence coverage. Retain source locators and identity bindings for every conclusion. Preserve missing, contradictory and unusable evidence without filling gaps from S4/S5 labels, hidden source records or general knowledge.
>
> Do not alter the historical archive or measured results. Do not call a model/API, start a framework workflow, run a test, execute a notebook kernel, operate the VM, or merge. Prepare implementation and a change account for review only. List unresolved specification questions before choosing a consequential interpretation. Return changed-file identities, requirement-to-code mappings, implementation limitations and the checks proposed for later approval. Preserve this assignment and your returned output exactly, with any redactions explicitly labeled.

Before dispatch, add the accepted specification commit, input hashes, assignment ID and actual dispatch time. Capture the actual return when it exists. Never backfill a reconstructed prompt as an exact historical exchange.

## 7. Review and next decision
**The next decision is acceptance of the bounded specification, followed by implementation preparation. Testing and merging remain separate approvals.**

| Question for review | Required answer |
|---|---|
| Is the evidence boundary precise? | Every allowed field, omission and source identity is declared. |
| Does the code implement the requirements? | Later source review maps each requirement to exact code and known gaps. |
| Do the examples distinguish the important cases? | Original and constructed examples stay separate; conflicts and positive evidence are preserved. |
| What may execute? | Only the specific checks later approved by Mark, with scope and expected records. |
| What would reject the component? | Unsupported confident findings, hidden answer leakage, lost source identity, silent repair, or conflated requester/authority/action status. |

A future check record should retain the exact command, interpreter/dependency versions, input and output digests, per-example outcomes, errors, review findings and limitations. Do not describe a source inspection as an executed check.

**What this POC would not establish:** effectiveness with human readers, live-model reliability, a complete research-team transcript, formal proof, native Temporal qualification, distributed revocation or production compliance. Those questions require their own evidence and authorization.

## Technical reference: source receipts and framework context
[Original archive at pinned revision](https://github.com/node-and-norm/revocation-delegated-workflows/blob/1866a73e61049c6e60354c0e4a565c8482736d33/archives/withdrawal_interface_study_v01_20261103.zip)

Archive SHA-256: `adf5e66cb0ad90038cde189330ee46877fabda26f1d829d3ae9b734c44af827c`

| Original member | SHA-256 |
|---|---|
| `measured01/framework/S4/record.json` | `265d275eaadf11b9537299fce72ea164df9d7fafa756a53b62e8cd48ef07ffce` |
| `measured01/framework/S5/record.json` | `304414e8e12dfaa39d650d3d8f87a9d6ce5f76946969a65db99c7288095cb665` |

These digests were obtained by reading the saved archive. Hashes identify bytes; they do not establish truth, completeness or independent witnessing. Original JSON member names and pointers in this notebook identify evidence inside the archive; they are not invented standalone GitHub file links.

[Versioned integration source](https://github.com/node-and-norm/revocation-delegated-workflows/blob/1866a73e61049c6e60354c0e4a565c8482736d33/integration/v0.3/workflow.py) shows actual LangChain agents/tools/HumanInTheLoopMiddleware and LangGraph checkpoint/resume machinery, using `StateScriptModel`. The measured archive's own frozen source remains authoritative for its historical cases. Later code must not silently replace that source.

[Research assignment workflow](https://github.com/node-and-norm/revocation-delegated-workflows/blob/1866a73e61049c6e60354c0e4a565c8482736d33/docs/team/WORKFLOW.md) · [Current safeguard notebook](https://github.com/node-and-norm/revocation-delegated-workflows/blob/1866a73e61049c6e60354c0e4a565c8482736d33/experiments/temporal-qualification-v0.1/05_safeguard_review_and_qualification_plan.ipynb) · [Operation/evidence contract](https://github.com/node-and-norm/revocation-delegated-workflows/blob/1866a73e61049c6e60354c0e4a565c8482736d33/experiments/temporal-qualification-v0.1/safeguards/OPERATION_EVIDENCE_CONTRACT.md)

The operation contract remains a proposed VM observation design. This report POC neither implements it nor resolves its gates.

[Spec-driven development paper](https://arxiv.org/abs/2609.00252) provides conceptual inspiration. The requirements, acceptance examples and report design here are our application, not results reported by its authors.

## Preparation and review receipt
This specification was prepared through read-only repository and archived-record inspection. The recorded S4 and S5 fields were compared with the proposed conclusions. No report implementation, acceptance check, notebook kernel, model-provider call or VM operation was run. No merge was performed.

**Independent AI source review:** a separate AI reviewer read the saved S4/S5 archive records and confirmed the event sequences and saved authority/effect states. The review required case-and-digest namespaces, explicit joins for denial identity, separation of inventory presence from capture completeness, protection against duplicate-evidence leakage, genuine same-boundary contradictions, retention of positive evidence under incomplete coverage and no classification by schedule label. Those corrections are incorporated above. This source review is not an executed component check or independent human validation.

Preserved review assignment, with the local archive locator explicitly redacted:
> Independently review prospective POC spec in my preceding final for evidence/identity errors, acceptance examples, overclaims. Read frozen source S4/S5 locally from [local archive locator redacted] if needed. No tests/implementation/repo writes. Return concrete corrections quickly. I'm authoring notebook spec only.

The repository archive link and member identities above provide the shareable source locators. It identifies an internal review task, not an experimental agent prompt. The findings above summarize the returned review; they are not labeled a verbatim transcript.

**Rendering:** after publication to the draft branch, the GitHub desktop preview was inspected at 1165 × 747 pixels. The opening roadmap and acceptance table rendered legibly; the preview loaded the notebook text. Mobile and native Jupyter views have not been checked. An earlier local HTML preview exported with Pandoc; local browser capture was blocked, and later GitHub inspection used the available cloud browser. No notebook kernel was executed.

The acceptance examples are prospective for the new component. The underlying S4/S5 observations are historical. The proposed implementer assignment remains **not dispatched**.